# The rejection region approach and the z-test
The two one-sample z-tests of the Note, and an interactive view of how alpha moves the rejection region.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy import stats

## 1. A one-sample z-test with the rejection region approach

In [ ]:
def z_test(x_bar, mu0, sigma, n, alpha=0.05, tail="two"):
    # z statistic: distance from mu0 in standard errors
    z = (x_bar - mu0) / (sigma / np.sqrt(n))
    # critical value(s) and the decision for each kind of H1
    if tail == "right":
        crit = stats.norm.ppf(1 - alpha)
        reject = z > crit
    elif tail == "left":
        crit = stats.norm.ppf(alpha)
        reject = z < crit
    else:
        crit = stats.norm.ppf(1 - alpha / 2)
        reject = abs(z) > crit
    return round(float(z), 3), round(float(crit), 3), "reject H0" if reject else "fail to reject H0"

# training program: H1 mu > 50
print(z_test(53, 50, 5, 30, tail="right"))
# chips packets: H1 mu != 50
print(z_test(49, 50, 4, 40, tail="two"))

## 2. Critical values for several alpha

In [ ]:
for a in [0.30, 0.10, 0.05, 0.01]:
    print(f"alpha {a:.2f}: one-tailed {stats.norm.ppf(1 - a):.3f}, two-tailed ±{stats.norm.ppf(1 - a / 2):.3f}")

## 3. Move alpha with the slider
The red tails are the rejection region of a two-tailed test.

In [ ]:
z = np.linspace(-4, 4, 600)
fig = go.Figure()
steps = []
alphas = [0.01, 0.05, 0.10, 0.20, 0.30]
for i, a in enumerate(alphas):
    c = stats.norm.ppf(1 - a / 2)
    tails = np.where(np.abs(z) >= c, stats.norm.pdf(z), np.nan)
    fig.add_scatter(x=z, y=stats.norm.pdf(z), line_color="black", visible=(i == 1), showlegend=False)
    fig.add_scatter(x=z, y=tails, fill="tozeroy", fillcolor="rgba(228,87,86,0.5)", line_width=0,
                    visible=(i == 1), showlegend=False)
    vis = [False] * (2 * len(alphas))
    vis[2 * i] = vis[2 * i + 1] = True
    steps.append(dict(method="update", label=f"{a:.2f}",
                      args=[{"visible": vis}, {"title": f"alpha = {a:.2f}: critical values ±{c:.2f}"}]))
fig.update_layout(template="simple_white", sliders=[dict(active=1, steps=steps, currentvalue={"prefix": "alpha "})],
                  title="alpha = 0.05: critical values ±1.96")
fig.show()

## 4. Why the rejection region approach hides the strength of evidence

In [ ]:
# upper-tail probability for z values that all lead to "reject"
for zv in [1.97, 2, 3.29, 15]:
    print(f"z = {zv:5}: P(Z >= z) = {stats.norm.sf(zv):.3g}")